# Lab 2: Lerobot Ex ROS

**Course:** TECHIN 517  
**Date:** 2026-04-15  
**Student:** Alan Nur

---
## Deliverable 1: Rosetta Contract

The following is the completed Rosetta contract for the SO101 robot arm (`soa_act_contract.yaml`).

```yaml
robot_type: soa

fps: 30
max_duration_s: 15.0

observations:
  - key: observation.images.wrist
    topic: /follower/image_raw
    type: sensor_msgs/msg/Image
    image:
      resize: [480, 640]
    align: {strategy: hold, stamp: header}
    qos: {reliability: best_effort, history: keep_last, depth: 10}

  - key: observation.images.overhead
    topic: /static_camera/overhead_cam/color/image_raw
    type: sensor_msgs/msg/Image
    image:
      resize: [720, 1280]
    align: {strategy: hold, stamp: header}
    qos: {reliability: best_effort, history: keep_last, depth: 10}

  - key: observation.state
    topic: /follower/joint_states
    type: sensor_msgs/msg/JointState
    selector:
      names:
        - position.shoulder_pan
        - position.shoulder_lift
        - position.elbow_flex
        - position.wrist_flex
        - position.wrist_roll
        - position.gripper
    align: {strategy: hold, stamp: header}
    qos: {reliability: best_effort, history: keep_last, depth: 50}
    unit_conversion: rad2deg

actions:
  - key: action
    publish:
      topic: /follower/arm_fwd_controller/commands
      type: std_msgs/msg/Float64MultiArray
      qos: {reliability: reliable, history: keep_last, depth: 10}
    selector:
      names:
        - position.shoulder_pan
        - position.shoulder_lift
        - position.elbow_flex
        - position.wrist_flex
        - position.wrist_roll
    unit_conversion: rad2deg
    safety_behavior: hold

  - key: action
    publish:
      topic: /follower/gripper_fwd_controller/commands
      type: std_msgs/msg/Float64MultiArray
      qos: {reliability: reliable, history: keep_last, depth: 10}
    selector:
      names:
        - position.gripper
    unit_conversion: rad2deg
    safety_behavior: hold

recording:
  storage: mcap
```

**Notes on contract design choices:**

- Two camera observations are used: a wrist camera (`/follower/image_raw`) and an overhead Intel RealSense (`/static_camera/overhead_cam/color/image_raw`), matching the two-camera setup the ACT model was trained on.
- `align: {strategy: hold, stamp: header}` is used for all observations to keep sensor data synchronized.
- Joint state observations use `best_effort` QoS with a larger depth (50) to avoid dropping fast-arriving state messages; actions use `reliable` QoS to ensure commands are delivered.
- The arm and gripper are published to separate forward controller topics as required by `soa_bringup`.
- `safety_behavior: hold` keeps the robot stationary when no policy is active.
- `unit_conversion: rad2deg` is applied to both observations and actions to match the SO101 hardware interface convention.

---
## Deliverable 2: Policy Deployment — Commands and Video

**Trained model:** `act_so101_2cam_v3-20260416T004131Z-3-002`  
**Task prompt:** `grab the aruco and put it on the yellow basket`  
**Camera setup:** wrist camera + overhead Intel RealSense

The following commands were used to deploy the policy through ROS 2 and Rosetta.

### Terminal 1 — Bring up the robot (forward-controller mode)

In [ ]:
%%bash
cd /home/ubuntu/techin517
source /opt/ros/humble/setup.bash
source /home/ubuntu/techin517/ros2_ws/install/setup.bash
ros2 launch soa_bringup soa_bringup.launch.py controller:=forward cameras:=true

### Terminal 2 — Start the controller switcher service

In [ ]:
%%bash
cd /home/ubuntu/techin517
source /opt/ros/humble/setup.bash
source /home/ubuntu/techin517/ros2_ws/install/setup.bash
ros2 run soa_functions controller_switcher

### Terminal 3 — Start Rosetta with the ACT policy

In [ ]:
%%bash
cd /home/ubuntu/techin517
source /opt/ros/humble/setup.bash
source /home/ubuntu/techin517/ros2_ws/install/setup.bash
ros2 launch rosetta rosetta_client_launch.py \
  contract_path:=/home/ubuntu/techin517/ros2_ws/src/soa_ros2/soa_bringup/rosetta_contracts/soa_act_contract.yaml \
  pretrained_name_or_path:=/home/ubuntu/techin517/act_so101_2cam_v3-20260416T004131Z-3-002/act_so101_2cam_v3/checkpoints/last/pretrained_model

### Terminal 4 — Send the policy goal

In [ ]:
%%bash
cd /home/ubuntu/techin517
source /opt/ros/humble/setup.bash
source /home/ubuntu/techin517/ros2_ws/install/setup.bash
ros2 action send_goal /run_policy \
  rosetta_interfaces/action/RunPolicy \
  "{prompt: 'grab the aruco and put it on the yellow basket'}"

### Video Demonstration

The video below shows the ROS 2 terminal output alongside the robot executing the policy autonomously.

[Watch on Google Drive](https://drive.google.com/file/d/1jHb2aREJCk1ruU6jeMcrzyrtSxvjEmDc/view?usp=sharing)

---
## Deliverable 3: Controller Switching Commands

### List active controllers

In [ ]:
%%bash
cd /home/ubuntu/techin517
source /opt/ros/humble/setup.bash
source /home/ubuntu/techin517/ros2_ws/install/setup.bash
ros2 control list_controllers --controller-manager /follower/controller_manager

### Switch: Forward Controllers → Joint Trajectory Controllers

In [ ]:
%%bash
cd /home/ubuntu/techin517
source /opt/ros/humble/setup.bash
source /home/ubuntu/techin517/ros2_ws/install/setup.bash
ros2 control switch_controllers \
  --controller-manager /follower/controller_manager \
  --deactivate arm_fwd_controller gripper_fwd_controller \
  --activate arm_controller gripper_controller

### Switch: Joint Trajectory Controllers → Forward Controllers

In [ ]:
%%bash
cd /home/ubuntu/techin517
source /opt/ros/humble/setup.bash
source /home/ubuntu/techin517/ros2_ws/install/setup.bash
ros2 control switch_controllers \
  --controller-manager /follower/controller_manager \
  --deactivate arm_controller gripper_controller \
  --activate arm_fwd_controller gripper_fwd_controller

### Switch using the Lab 2 `controller_switcher` service

Switch to forward controllers:

In [ ]:
%%bash
cd /home/ubuntu/techin517
source /opt/ros/humble/setup.bash
source /home/ubuntu/techin517/ros2_ws/install/setup.bash
ros2 service call /controller_switcher/switch_controller \
  soa_interfaces/srv/SwitchController \
  "{controller_type: forward}"

Switch to joint trajectory controllers:

In [ ]:
%%bash
cd /home/ubuntu/techin517
source /opt/ros/humble/setup.bash
source /home/ubuntu/techin517/ros2_ws/install/setup.bash
ros2 service call /controller_switcher/switch_controller \
  soa_interfaces/srv/SwitchController \
  "{controller_type: jtc}"

---
## Deliverable 4: Reflection

When running a lerobot policy through Rosetta, I found that using different controllers really matters depending on what kind of task you're doing. The ForwardCommandController is simple, it just takes a value and sends it straight to the motor, no questions asked. This works great for lerobot because the policy outputs raw joint positions at every timestep and needs them applied immediately without any extra processing in between. If I used a JointTrajectoryController here, it would try to smooth and interpolate between points, which would fight against the policy's own timing and make the motion laggy or unpredictable. On the other hand, when I want to do classical motion planning with something like MoveIt, the JointTrajectoryController is much better because it handles smooth interpolation, velocity limits, and timing automatically,  things that matter when you're following a planned path safely. So the reason to switch controllers is basically: forward controllers get out of the way and let the policy drive, while trajectory controllers add the structure that classical planners expect.